# 실습: Agent Capability 평가와 Debugging

이 노트북은 Day04_new의 여섯 번째 노트북입니다. 앞에서 만든 요구사항 인터뷰 MCP 서버(도구 7개)와 Deep Agent가 **능력 수준에서 어디서 틀리는지**를, 사람의 감이 아니라 코드가 세는 점수로 가르는 방법을 연습합니다. 모델이 판정하는 평가(LLM judge)는 쓰지 않습니다. 이 노트북의 점수는 모두 `cap_lab/scorers.py`의 결정적 함수가 호출 기록과 근거(Evidence) 파일을 읽어 계산합니다.

### 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 0. 준비 | 11개 검사와 숫자 5개 지도, 손으로 만든 기록 도구 | 8 |
| 1. Tool 선택과 Argument Correctness | tool_selection_ok, args_valid, None의 의미 | 10 |
| 2. Trigger Eval과 Quality Eval | trigger_ok, frozen_ok, vague_not_frozen을 따로 읽기 | 10 |
| 3. 권한 거부 · 오류 · 재시도 · 중복 실행 | no_blind_retry, retry_bounded, side_effect_stable, contamination_free | 10 |
| 4. 디버깅 걷기 | 실패한 점수에서 호출과 근거 파일까지 | 5 |
| 5. 실제 실행 5회 | dev 사례 5개를 기준선으로 실행하고 근거와 대조 | 12 |
| 6. 직접 해보기 | 내 사례 만들기와 경계 실험 | 5 |

### 이 노트북이 하는 일 (그리고 하지 않는 일)

- 하는 일: 11개 검사가 각각 **언제 적용되고, 무엇을 세고, 어디서 None이 되는지** 손으로 만든 기록으로 확인합니다. 그다음 실제 Deep Agent를 dev 사례 5개로 한 번씩 돌려서 같은 점수표를 읽고, 실패한 점수에서 호출과 근거 파일까지 거슬러 올라갑니다.
- 하지 않는 일: 에이전트를 고치지 않습니다(개선과 측정은 07의 일입니다). Langfuse에는 아무것도 보내지 않습니다. 이 노트북의 실행 기록은 모두 이 컴퓨터의 `outputs/notebook/` 아래에만 남습니다. 점수를 매겨 순위를 정하지도 않습니다. 이 저장소의 모든 실습처럼 여기서도 결과는 학습용 관찰입니다.

### 호출 규모

- 0~4절과 직접 해보기의 기본 셀: 유료 호출이 없습니다. Claude도 Jev도 부르지 않습니다. 손으로 만든 기록과, 모델 없이 도구만 호출하는 MCP 서버(자식 프로세스)만 씁니다.
- 5절의 실제 실행 셀 하나: Claude 약 45회 이하, 실행 5회, 2~5분입니다. 사례당 Claude 호출 5~9회를 상한으로 잡은 추정입니다(`cap_lab/experiment.py:52`). 게이트를 끈 기준선이므로 Jev는 부르지 않습니다. 실행 직전에 셀이 같은 내용을 한 줄로 다시 알립니다. 이 비용은 본인의 Anthropic 크레딧에서 나갑니다.
- 직접 해보기의 `RUN_EXERCISE`, `RUN_OPTIONAL` 셀은 기본값이 `False`라서 켜지 않으면 아무것도 부르지 않습니다.

### 표지 규칙

코드 셀 첫 줄의 표지는 하는 일을 알려 줍니다. `# [따라하기]`는 그대로 실행, `# [바꿔보기]`는 값을 바꿔 다시 실행, `# [깨뜨리기]`는 일부러 점수를 속이거나 규칙의 빈틈을 찾는 셀, `# [내 것 만들기]`는 내 사례를 직접 쓰는 셀입니다. `**먼저 예측하세요.**`가 나오면 실행하기 전에 결과를 한 줄이라도 적어 보세요. 맞히는 것이 목적이 아니라, 틀렸을 때 어느 가정이 틀렸는지 찾는 것이 목적입니다.

## 0. 준비: 11개 검사와 숫자 5개

한 번의 에이전트 실행이 끝나면 `RunRecord`가 남습니다. 호출 기록(`tool_calls`), 읽은 Skill 경로(`skill_reads`), 서버가 근거 폴더에 쓴 파일(`evidence_dir`), 모델 호출 수와 토큰, 오류(`error`)가 들어 있습니다(`cap_lab/scorers.py:43`). 점수 함수는 이 기록 하나만 받아서 `Score(name, value, comment)`를 돌려줍니다. 값은 세 가지입니다.

- `True`: 이 검사가 적용되었고 기대와 같았다.
- `False`: 이 검사가 적용되었고 기대와 달랐다.
- `None`: **검사하지 않았거나 이 사례에는 적용되지 않는다.** 실패가 아닙니다.

세 번째가 이 노트북의 가장 큰 주제 중 하나입니다. 사례마다 기대(`expect`)와 장애(`fault`)가 다르기 때문에, 한 검사가 적용되는 사례는 일부뿐입니다. `None`을 `False`와 같이 취급하면 통과율이 거짓으로 낮아집니다. 반대로 `None`을 빼고 세면서 얼마나 많은 실행을 못 봤는지 말하지 않으면 통과율이 거짓으로 높아 보입니다. 그래서 이 저장소의 표는 통과 수, 확인한 수, 확인하지 못한 수를 함께 보여 줍니다.

아래 셀에서 먼저 도구를 가져오고, 커널의 작업 폴더가 `Day04_new`인지 확인합니다. `key_table()`은 키가 설정되었는지만 보여 주고 값은 보여 주지 않습니다.

In [ ]:
# [따라하기] 가져오기와 작업 폴더 확인
import inspect
import json
import tempfile
from itertools import count
from pathlib import Path

from cap_lab.experiment import Variant, load_cases, run_case
from cap_lab.notebook import (
    announce, connect_server, key_table, new_workdir, pass_rate_table, show_events, show_table,
)
from cap_lab.scorers import (
    CHECKS, MCP_TOOLS, STAGE_OF, RunRecord, args_valid, contamination_free,
    first_failure, frozen_ok, no_blind_retry, retry_bounded, score_run,
    side_effect_stable, tool_selection_ok, trigger_ok, vague_not_frozen,
)
from cap_lab.server.evidence import EvidenceStore
from cap_lab.server.server import FaultPlan
from cap_lab.server.workflow import Workflow
from cap_lab.settings import load_settings

print("커널 작업 폴더:", Path.cwd().name)
assert Path("cap_lab").is_dir(), "커널의 작업 폴더가 Day04_new여야 합니다."
key_table()

### 11개 검사와 단계

`cap_lab/scorers.py:293`의 `CHECKS`가 11개 검사의 순서입니다. 단계는 `STAGE_OF`(`cap_lab/scorers.py:28`)에 있습니다. S1 Skill 트리거, S2 시작 인자(도구 선택·인자·목표 유지), S3 모호한 답변을 명확히 함, S5 결과 확정(freeze), S6 근거 연결이고, R1 권한 거부, R2 일시 오류, R3 중복 호출은 견고성 검사입니다. S4는 동의 단계였다가 제거되어 번호만 비어 있습니다. `contamination_free`는 `STAGE_OF`에 없습니다. 어느 단계에도 속하지 않고 모든 실행에 적용되기 때문입니다.

아래 표의 "적용 조건"은 각 함수 앞부분의 `None`을 돌려주는 분기를 읽고 제가 정리한 것입니다. 표를 믿지 말고, 바로 다음 셀들에서 실제 dev 사례 10개로 맞는지 확인합니다. 문서와 코드가 다르면 코드가 정답입니다.

In [ ]:
# [따라하기] 11개 검사의 단계, 적용 조건, 코드 위치
# (설명, 사례만 보고 적용 여부를 말해 주는 함수). args_valid와 contamination_free는 사례가 아니라 기록을 본다.
APPLIES_WHEN = {
    "trigger_ok": ("expect.skill_read가 있다(참 또는 거짓)", lambda case: case["expect"].get("skill_read") is not None),
    "tool_selection_ok": ("expect.must_call 또는 must_not_call이 비어 있지 않다",
                          lambda case: bool(case["expect"].get("must_call") or case["expect"].get("must_not_call"))),
    "args_valid": ("기록에 MCP 도구 호출이 하나라도 있다", lambda case: True),
    "goal_preserved": ("expect.goal_verbatim이 있다", lambda case: case["expect"].get("goal_verbatim") is not None),
    "vague_not_frozen": ("expect.vague_answer가 있다", lambda case: case["expect"].get("vague_answer") is not None),
    "frozen_ok": ("expect.frozen이 있다(참 또는 거짓)", lambda case: case["expect"].get("frozen") is not None),
    "evidence_linked": ("expect.evidence_linked가 있다", lambda case: case["expect"].get("evidence_linked") is not None),
    "no_blind_retry": ("fault.permission_denied가 있다", lambda case: bool((case.get("fault") or {}).get("permission_denied"))),
    "retry_bounded": ("fault.transient가 있다", lambda case: bool((case.get("fault") or {}).get("transient"))),
    "side_effect_stable": ("expect.event_count가 있다", lambda case: case["expect"].get("event_count") is not None),
    "contamination_free": ("항상 적용된다(실행 오류가 있으면 통과는 None이 된다)", lambda case: True),
}
CHECK_NAMES = [check.__name__ for check in CHECKS]
assert set(CHECK_NAMES) == set(APPLIES_WHEN), "검사 이름이 표와 다릅니다."

rows = []
for check in CHECKS:
    line = inspect.getsourcelines(check)[1]  # 함수가 시작하는 줄 번호
    rows.append({
        "검사": check.__name__,
        "단계": STAGE_OF.get(check.__name__, "전체"),
        "적용 조건": APPLIES_WHEN[check.__name__][0],
        "코드": f"cap_lab/scorers.py:{line}",
    })
show_table(rows)

**관찰할 것.** 단계 칸을 위에서 아래로 읽으면 S1, S2, S2, S2, S3, S5, S6, R1, R2, R3, 전체 순서입니다. 같은 S2 안에 도구 선택, 인자, 목표 유지가 세 검사로 나뉘어 있습니다. 하나의 "도구를 잘 썼는가"를 세 가지 다른 질문으로 쪼갰기 때문에, 실패했을 때 어느 질문에서 실패했는지 바로 알 수 있습니다. 적용 조건 칸에서는 대부분이 "expect에 ...가 있다"입니다. 즉 **사례 파일이 무엇을 기대하는지 적어 둔 만큼만** 검사가 켜집니다.

### 손으로 만든 기록

앞으로 자주 쓸 도구를 만듭니다. 이 노트북의 "손으로 만든 기록"은 **가짜 모델이 아니라 데이터**입니다. 모델이 호출했다고 주장하지 않고, 이런 호출 기록이 있었다면 점수 함수가 어떻게 반응하는지만 봅니다. 그래서 호출 기록(`tool_calls`)은 사전으로 직접 쓰고, 근거 폴더는 **서버가 쓰는 같은 업무 코드**(`Workflow`와 `EvidenceStore`)를 직접 호출해서 만듭니다. 파일 모양을 손으로 흉내 내면 점수 함수가 읽는 필드와 어긋날 수 있기 때문입니다.

- `make_case`: `data/cases.jsonl`과 같은 키를 가진 사례 사전을 만든다. **파일에 쓰지 않고** 이 노트북 안에서만 쓴다.
- `new_lab`, `interview`: 임시 폴더에 서버와 같은 코드로 세션을 만든다(`cap_lab/server/workflow.py:171`의 `start_interview`, `:180`의 `record_answer`, `:218`의 `freeze_seed`).
- `tool_call`, `make_record`: 호출 하나와 `RunRecord` 하나를 만든다.
- `show_check`: 한 검사를 여러 기록에 적용해서 값과 이유(comment)를 나란히 보여 준다.

In [ ]:
# [따라하기] 손으로 만든 기록 도구 (데이터를 만들 뿐, 모델을 흉내 내지 않는다)
GOAL = "회의실 예약이 잡히면 참석자에게 알림 메일을 보내는 기능"
VAGUE = "분류가 잘 되면 완료"
CLEAR = "샘플 문의 50건 중 담당 분류와 일치하는 문의가 45건 이상이면 완료"
SKILL_PATH = "/requirements-interview/SKILL.md"  # 점수 함수는 이 경로의 끝부분만 본다(cap_lab/scorers.py:15)

_call_numbers = count(1)


def make_case(case_id, expect, *, fault=None, stage="S1", kind="hand_made"):
    """손으로 만든 사례 사전. data/cases.jsonl의 사례와 같은 키를 갖는다. 파일에는 쓰지 않는다."""
    return {"id": case_id, "split": "dev", "stage": stage, "kind": kind, "turns": ["(손으로 만든 사례)"],
            "elicitation": {}, "fault": fault, "requires": None, "expect": expect}


def new_lab(label):
    """임시 폴더에 서버와 같은 업무 코드(Workflow, EvidenceStore)를 만든다. (workflow, 근거 폴더)를 돌려준다."""
    root = Path(tempfile.mkdtemp(prefix=f"lab06-{label}-"))
    store = EvidenceStore(root / "evidence")
    return Workflow(root / "sessions.sqlite3", store), store.root


def interview(workflow, goal, *, constraints=None, criteria=None, freeze=False):
    """세션 하나를 시작하고 답을 기록한다. freeze=True면 명세까지 확정한다. 세션 ID를 돌려준다."""
    session_id = workflow.start_interview(goal)["session_id"]
    if constraints is not None:
        workflow.record_answer(session_id, "constraints", constraints)
    if criteria is not None:
        workflow.record_answer(session_id, "acceptance_criteria", criteria)
    if freeze:
        workflow.freeze_seed(session_id)
    return session_id


def empty_evidence():
    """세션이 하나도 없는 근거 폴더."""
    return Path(tempfile.mkdtemp(prefix="lab06-empty-"))


def tool_call(name, args=None, *, result=None, ok=True):
    """RunRecord.tool_calls의 항목 하나. 실제 실행이 남기는 키(id, name, args, result, ok)와 같다."""
    return {"id": f"hand-{next(_call_numbers)}", "name": name, "args": args or {},
            "result": {"success": True} if result is None else result, "ok": ok}


def make_record(case, evidence_dir, calls=(), skill_reads=(), error=None, **numbers):
    """손으로 만든 RunRecord. 숫자(model_calls 등)는 키워드로 넘긴다."""
    return RunRecord(case=case, evidence_dir=evidence_dir, tool_calls=list(calls),
                     skill_reads=list(skill_reads), error=error, **numbers)


def show_check(check, labelled):
    """한 검사를 이름 붙인 기록들에 적용해 값과 이유를 표로 보여 준다."""
    rows = []
    for label, record in labelled.items():
        score = check(record)
        rows.append({"손으로 만든 기록": label, "값": score.value, "이유(comment)": score.comment})
    show_table(rows)


print("도구 준비 완료:", [make_case.__name__, new_lab.__name__, interview.__name__, tool_call.__name__,
                       make_record.__name__, show_check.__name__])

### 적용 조건 표를 dev 사례 10개로 확인하기

위 표가 맞다면, 사례만 보고 "이 검사는 켜진다/꺼진다"를 말할 수 있어야 합니다. 아래 셀은 **dev 사례 10개마다** 기록을 하나씩 만들고(MCP 도구 호출 한 번, 근거 폴더는 비어 있음), 11개 검사 중 값이 `None`이 아닌 것을 모아서 표의 예측과 비교합니다. 불일치가 있으면 표가 틀린 것이므로 코드를 다시 읽어야 합니다. 이 비교가 맞는다는 것은 `None`이 "사례가 그 검사를 요구하지 않음"에서 나온다는 뜻이지, 실행이 잘되었다는 뜻이 아닙니다.

In [ ]:
# [따라하기] 적용 조건 표 검증: 예측(표)과 관찰(점수 함수)의 불일치를 센다
dev_cases = load_cases(split="dev")
one_mcp_call = tool_call("get_session", {"session_id": "0" * 32})

mismatches = []
applied_table = []
for case in dev_cases:
    probe = make_record(case, empty_evidence(), calls=[one_mcp_call])
    observed = {score.name: score.value is not None for score in score_run(probe) if score.name in APPLIES_WHEN}
    predicted = {name: rule(case) for name, (_, rule) in APPLIES_WHEN.items()}
    mismatches += [(case["id"], name) for name in observed if observed[name] != predicted[name]]
    applied_table.append({"사례": case["id"], "단계": case["stage"],
                          "적용되는 검사": ", ".join(name for name, on in observed.items() if on)})

show_table(applied_table)
print("표와 코드가 다른 (사례, 검사):", mismatches)

**관찰할 것.** `불일치` 목록이 비어 있어야 합니다. 그리고 표를 읽어 보세요. dev-02(Skill을 쓰지 말아야 하는 사례)에서는 `trigger_ok`와 `tool_selection_ok`가 켜집니다. 두 검사 모두 "하지 않은 것"을 확인합니다. 사례가 하나의 단계만 겨냥하는 것처럼 보여도 `args_valid`와 `contamination_free`는 모든 사례에서 켜집니다. dev-09와 dev-10은 `fault`가 있어서 각각 R1(`no_blind_retry`), R2(`retry_bounded`) 검사가 추가로 켜집니다.

### 숫자 5개

`score_run`(`cap_lab/scorers.py:299`)은 11개 검사 뒤에 숫자 5개를 붙입니다. `model_calls`, `tool_calls`, `latency_s`, `input_tokens`, `output_tokens`입니다. 이 숫자들은 참/거짓이 아니라 크기라서 통과율로 세지 않습니다(`pass_rate_table`은 숫자 점수를 넘기면 오류를 냅니다). 비용과 속도의 가드 지표로 따로 읽습니다.

아래 셀은 모델 호출 3회, 도구 호출이 없는 손으로 만든 기록 하나에 `score_run`을 적용합니다. 값이 `None`인 줄의 `comment`에 **왜 None인지**가 한국어로 적혀 있는 것을 확인하세요.

In [ ]:
# [따라하기] score_run이 돌려주는 16개 점수: 검사 11개 + 숫자 5개
bare_case = make_case("hand-bare", {})  # 기대가 하나도 없는 사례
bare_record = make_record(bare_case, empty_evidence(), model_calls=3, input_tokens=1200,
                          output_tokens=150, latency_s=12.5)
bare_scores = score_run(bare_record)

show_table([{"점수": s.name, "값": s.value, "단계": STAGE_OF.get(s.name, "-"), "이유(comment)": s.comment}
            for s in bare_scores])
print("점수 개수:", len(bare_scores), "= 검사", len(CHECKS), "+ 숫자", len(bare_scores) - len(CHECKS))
print("first_failure:", first_failure(bare_scores))

**관찰할 것.** 기대가 하나도 없는 사례에서 10개 검사는 `None`입니다. `args_valid`는 MCP 호출이 없다는 이유로 `None`, `contamination_free`만 `True`입니다. 살펴볼 점은 `first_failure`가 `None`이라는 것입니다. "실패 없음"이 아니라 **"실패로 확인된 것이 없음"** 입니다. 아무것도 확인하지 않은 실행도 같은 결과를 냅니다. 이 차이를 놓치면 "점수가 다 깨끗하다"는 말을 믿게 됩니다. 숫자 5개는 `Score.value`가 `float`입니다. `model_calls`가 `3.0`으로 보이는 이유입니다.

## 1. Tool 선택과 Argument Correctness

같은 "도구를 잘 쓰는가"에도 서로 다른 실패가 있습니다.

1. **도구 선택**: 불러야 할 도구를 안 부르거나, 부르면 안 되는 도구를 부른다(`tool_selection_ok`).
2. **인자 정확성**: 도구는 맞게 골랐는데 인자가 틀렸다(`args_valid`).

`tool_selection_ok`(`cap_lab/scorers.py:134`)는 호출된 도구 **이름의 집합**만 봅니다. 사례의 `must_call`에 있는 이름이 하나라도 빠지거나 `must_not_call`에 있는 이름이 하나라도 있으면 실패입니다. 호출 순서와 횟수, 인자는 보지 않습니다. 이름만 보기 때문에 "맞는 도구를 잘못된 인자로 불렀다"는 이 검사에서는 통과입니다. 그 빈틈을 `args_valid`(`cap_lab/scorers.py:152`)가 메웁니다. 이 함수는 두 가지를 봅니다.

- **입력 검증 오류**: MCP 서버가 인자 형식이 틀렸다고 거절한 호출이 있는가. 이 오류는 도구 자체가 실행되기 전에 SDK 단계에서 납니다.
- **세션 ID가 서버가 만든 것인가**: `session_id` 인자가 근거 폴더에 실제로 있는 세션과 같은가. 모델이 ID를 지어내는 실패(hallucinated handle)를 잡습니다.

**먼저 예측하세요.** 아래 셀은 `tool_selection_ok`를 손으로 만든 기록 네 개에 적용합니다. (1) 필수 도구를 불렀다, (2) 필수 도구를 안 불렀다(도구 없이 글로만 답했다), (3) 부르면 안 되는 도구를 불렀다(Skill을 쓰지 말아야 하는 질문에서 인터뷰를 시작했다), (4) 사례에 도구 기대가 없다. 각각 `True`, `False`, `None` 중 무엇일지 적고 실행하세요.

In [ ]:
# [따라하기] tool_selection_ok: 통과, 실패(안 부름), 실패(금지 도구), None
must_start = make_case("hand-S2-a", {"must_call": ["start_interview"]})
no_tools_allowed = make_case("hand-S1-b", {"skill_read": False, "must_not_call": ["start_interview"]})
only_skill_expected = make_case("hand-S1-c", {"skill_read": True})  # 도구 기대가 없다

start_call = tool_call("start_interview", {"goal": GOAL})
labelled = {
    "통과: 필수 도구 start_interview를 부름": make_record(must_start, empty_evidence(), calls=[start_call]),
    "실패: 필수 도구를 안 부름(글로만 답함)": make_record(must_start, empty_evidence(), calls=[]),
    "실패: 부르면 안 되는 도구를 부름": make_record(no_tools_allowed, empty_evidence(), calls=[start_call]),
    "None: 사례에 도구 기대가 없음": make_record(only_skill_expected, empty_evidence(), calls=[start_call]),
}
show_check(tool_selection_ok, labelled)

**관찰할 것.** 첫 줄과 넷째 줄은 같은 `start_interview` 호출을 담고 있지만 결과가 `True`와 `None`으로 갈립니다. 호출 기록이 아니라 **사례의 기대**가 검사를 켰기 때문입니다. 둘째 줄의 이유(comment)에는 "호출해야 하는데 안 부른 도구"가 이름과 함께 적힙니다. 점수만 보고 고치는 것이 아니라 이유 문장에서 시작하는 것이 디버깅의 첫 걸음입니다.

### 인자 오류는 실제로 어떤 모양인가

`args_valid`가 입력 검증 오류를 알아보는 방법은 결과(`result`)가 어떤 모양인지에 달려 있습니다. 이 모양을 **상상해서 쓰면 안 됩니다.** 실제 MCP 서버에 일부러 틀린 인자를 보내 보면 됩니다. 아래 셀은 모델 없이 `connect_server`로 서버를 띄우고(`cap_lab/notebook.py`) `start_interview`에 문자열이 아닌 숫자 `123`을 목표로 보냅니다. 유료 호출이 아니고, 서버는 이 폴더의 `cap_lab/server/server.py`입니다.

에이전트 경로에서는 같은 오류가 "텍스트 블록 하나가 든 목록"이고 `ok=False`로 기록됩니다(`tests/test_realism.py:52-57`이 실제 서버와 어댑터를 거쳐 확인한 모양). 업무 오류(예: 없는 세션 ID)는 다른 모양입니다. `success: false`와 `error` 코드를 가진 **사전**이고 역시 `ok=False`입니다. 두 모양을 구분해야 하는 이유는 입력 검증 오류만 `args_valid`의 실패 원인이기 때문입니다.

In [ ]:
# [따라하기] 실제 서버에 틀린 인자를 보내 오류의 모양을 본다 (모델 없음, 유료 호출 없음)
shape_workdir = Path(tempfile.mkdtemp(prefix="lab06-shape-"))
async with connect_server(shape_workdir) as client:
    bad_type = await client.call_tool("start_interview", {"goal": 123})           # 형식 오류
    unknown_id = await client.call_tool("get_session", {"session_id": "a" * 32})  # 업무 오류

print("[형식 오류] is_error =", bad_type.is_error, "/ structured_content =", bad_type.structured_content)
validation_text = bad_type.content[0].text
print(validation_text)
print()
print("[업무 오류] is_error =", unknown_id.is_error, "/ structured_content =", unknown_id.structured_content)

**관찰할 것.** 형식 오류는 `is_error=True`이고 구조화된 결과(`structured_content`)가 없습니다. 본문은 `Error executing tool start_interview: 1 validation error for start_interviewArguments`로 시작하고 `[type=string_type`이 들어 있습니다. 업무 오류는 `is_error=False`이면서 `structured_content`가 `success: false`와 `error: "session_not_found"`를 담습니다. 도구는 실행되었고 그 결과가 "실패"라고 말하는 것입니다. 에이전트의 `ok` 필드(`cap_lab/experiment.py:271`의 `_record_tool_message`)는 두 경우 모두 `False`로 만들지만, `args_valid`는 `result`가 사전이 아닌 경우만 검증 오류로 읽습니다.

이제 이 실제 문구를 손으로 만든 기록의 `result`에 넣어 `args_valid`의 통과, 실패, None을 모두 봅니다.

In [ ]:
# [따라하기] args_valid: 통과, 실패(검증 오류), 실패(없는 세션 ID), None(MCP 호출 없음)
workflow, evidence_dir = new_lab("args")
real_session = interview(workflow, GOAL)  # 서버가 만든 진짜 세션 ID
args_case = make_case("hand-S2-d", {"must_call": ["start_interview"]})

# 에이전트 경로의 모양: 텍스트 블록 하나가 든 목록, ok=False (위에서 본 실제 문구를 그대로 넣는다)
validation_result = [{"type": "text", "text": validation_text}]
labelled = {
    "통과: 진짜 세션 ID로 record_answer": make_record(args_case, evidence_dir, calls=[
        tool_call("start_interview", {"goal": GOAL}),
        tool_call("record_answer", {"session_id": real_session, "field": "constraints", "answer": "없음"})]),
    "실패: 입력 검증 오류가 난 호출": make_record(args_case, evidence_dir, calls=[
        tool_call("start_interview", {"goal": 123}, result=validation_result, ok=False)]),
    "실패: 서버가 만들지 않은 세션 ID": make_record(args_case, evidence_dir, calls=[
        tool_call("get_session", {"session_id": "a" * 32},
                  result={"success": False, "error": "session_not_found"}, ok=False)]),
    "None: MCP 호출이 하나도 없음(read_file만)": make_record(args_case, evidence_dir, calls=[
        tool_call("read_file", {"file_path": SKILL_PATH}, result="(SKILL.md 본문)")]),
}
show_check(args_valid, labelled)

`args_valid`의 검증 오류 판정은 세 조건이 **모두** 필요합니다(`cap_lab/scorers.py:156-165`). `ok`가 `False`이고, `result`가 사전이 아니고, 본문에 `validation error`가 들어 있어야 합니다. 이유는 코드 주석에 있습니다. 문구만 보면 성공한 결과의 본문이 그 문구를 인용했을 때도 실패로 잡히고, 구조화 신호만 보면 검증이 아닌 다른 도구 오류도 섞입니다. 그렇다면 이 가정이 깨지는 모양에서는 어떻게 될까요?

**먼저 예측하세요.** 위의 "실패: 입력 검증 오류" 기록에서 (a) `ok`만 `None`으로 바꾸면, (b) `result`를 같은 문구가 든 **사전**으로 바꾸면 `args_valid`는 각각 무엇을 돌려줄까요? 이 값들이 `False`가 아니라면 점수 함수가 실제 오류를 놓치는 것입니다.

In [ ]:
# [바꿔보기] 검증 오류 기록의 모양을 바꿔서 args_valid가 어떻게 반응하는지 본다
def validation_variant(*, ok, result):
    """검증 오류를 담은 호출 하나짜리 기록. ok와 result의 모양만 바꾼다."""
    call = tool_call("start_interview", {"goal": 123}, result=result, ok=ok)
    return make_record(args_case, evidence_dir, calls=[call])


labelled = {
    "원래 모양: ok=False, 결과는 텍스트 블록 목록": validation_variant(ok=False, result=validation_result),
    "(a) ok=None으로 바꿈": validation_variant(ok=None, result=validation_result),
    "(b) 결과를 같은 문구가 든 사전으로 바꿈": validation_variant(ok=False, result={"text": validation_text}),
    "(c) 결과를 문자열로 바꿈(ok=False)": validation_variant(ok=False, result=validation_text),
}
show_check(args_valid, labelled)

**관찰할 것.** (a)와 (b)는 `True`로 바뀌어 검증 오류를 놓칩니다. (c)처럼 문자열이면 잡힙니다. `args_valid`는 "에이전트 경로가 오류를 이 모양으로 기록한다"는 가정 위에 서 있습니다. 가정이 틀리면 실패가 통과로 둔갑합니다. 그래서 이 저장소는 점수 함수를 만든 뒤 가정을 **실제 서버와 어댑터를 거친 기록으로** 확인하는 시험(`tests/test_realism.py`)을 따로 두었습니다. 평가 코드도 코드이고, 평가 코드의 가정도 검증 대상입니다.

### None을 실패로 세면 무슨 일이 생기는가

이제 첫 번째로 규칙을 일부러 깨 봅니다. 아래 셀은 `tool_selection_ok`를 여섯 개 기록에 적용합니다. 세 개는 도구 기대가 있는 사례(통과 둘, 실패 하나)이고 세 개는 도구 기대가 없는 사례(`None`)입니다. 이 결과를 두 가지 방식으로 요약합니다.

- **나이브**: `None`을 `False`로 취급하고 전체 개수로 나눈다.
- **이 저장소의 방식**: `None`은 센 칸에서 빼고, 확인한 수와 확인하지 못한 수를 따로 보여 준다. 표본이 작으므로 Wilson 95% 구간(`cap_lab/stats.py`)도 같이 본다.

**먼저 예측하세요.** 통과율이 얼마씩 나올까요? 그리고 두 숫자 중 어느 쪽이 "에이전트가 도구를 못 고른다"는 주장의 근거가 될 수 있을까요?

In [ ]:
# [깨뜨리기] None을 실패로 세면 통과율이 어떻게 움직이는가
with_expect = make_case("hand-S2-e", {"must_call": ["start_interview"]})
without_expect = make_case("hand-S1-f", {"skill_read": True})

records = [
    make_record(with_expect, empty_evidence(), calls=[start_call]),        # 통과
    make_record(with_expect, empty_evidence(), calls=[start_call]),        # 통과
    make_record(with_expect, empty_evidence(), calls=[]),                  # 실패
    make_record(without_expect, empty_evidence(), calls=[start_call]),     # None
    make_record(without_expect, empty_evidence(), calls=[]),               # None
    make_record(without_expect, empty_evidence(), calls=[start_call]),     # None
]
values = [tool_selection_ok(record).value for record in records]
print("tool_selection_ok 값:", values)

naive_rate = values.count(True) / len(values)  # None도 분모에 넣는다 = None을 실패로 센 것
print(f"나이브(None을 실패로 셈): {values.count(True)}/{len(values)} = {naive_rate:.2f}")

rows = [{"variant": "hand", "tool_selection_ok": value} for value in values]
show_table(pass_rate_table(rows, ["tool_selection_ok"]))

**관찰할 것.** 나이브 방식은 `0.33`, 이 저장소의 방식은 확인한 3건 중 2건 통과로 `0.67`입니다. 같은 여섯 기록에서 숫자가 두 배로 갈립니다. 둘 다 틀린 숫자는 아닙니다. 하지만 **무엇을 분모로 했는지 말하지 않으면** 서로 다른 질문의 답입니다. 이 저장소의 표는 `checked`(확인한 수)와 `unchecked`(확인하지 못한 수)를 항상 같이 보여 줍니다. 구간 `ci_low`~`ci_high`가 아주 넓은 것도 보세요. 3건으로는 거의 아무것도 말할 수 없습니다.

## 2. Trigger Eval과 Quality Eval을 나누기

Skill을 쓰는 에이전트에게는 서로 다른 두 질문이 있습니다.

- **Trigger Eval**: 이 요청에서 Skill을 **읽어야 했는가, 읽었는가.** 읽어야 하는 요청(dev-01)과 읽지 말아야 하는 요청(dev-02)이 둘 다 필요합니다. 이 저장소에서는 `trigger_ok`(S1)입니다.
- **Quality Eval**: Skill을 읽은 뒤(혹은 안 읽고도) **결과가 좋았는가.** 모호한 완료 기준이 확정 명세가 되지 않았는가(`vague_not_frozen`, S3), 명세가 확정되었는가(`frozen_ok`, S5), 산출물이 근거에 연결되었는가(`evidence_linked`, S6)입니다.

두 질문은 독립입니다. 네 가지 조합이 모두 가능합니다.

| | 결과가 좋음 | 결과가 나쁨 |
|---|---|---|
| **Skill을 읽음** | 이상적 | 읽었는데도 모호한 기준을 확정함 |
| **Skill을 안 읽음** | 안 읽고도 끝냄 | 둘 다 실패 |

`trigger_ok`(`cap_lab/scorers.py:123`)는 `skill_reads`의 경로 중 하나가 `requirements-interview/SKILL.md`로 끝나는지만 봅니다. `expect.skill_read`가 `True`면 읽어야 통과, `False`면 읽지 않아야 통과입니다. 먼저 trigger_ok의 통과, 실패, None을 다섯 기록으로 확인합니다.

In [ ]:
# [따라하기] trigger_ok: should_trigger와 should_not_trigger 양쪽의 통과, 실패, None
should_trigger = make_case("hand-S1-g", {"skill_read": True, "must_call": ["start_interview"]})
should_not_trigger = make_case("hand-S1-h", {"skill_read": False, "must_not_call": ["start_interview"]})
no_trigger_expectation = make_case("hand-S2-i", {"must_call": ["start_interview"]})

labelled = {
    "통과: 읽어야 하는 요청에서 Skill을 읽음": make_record(should_trigger, empty_evidence(), skill_reads=[SKILL_PATH]),
    "실패: 읽어야 하는 요청에서 Skill을 안 읽음": make_record(should_trigger, empty_evidence(), skill_reads=[]),
    "통과: 읽지 말아야 하는 요청에서 안 읽음": make_record(should_not_trigger, empty_evidence(), skill_reads=[]),
    "실패: 읽지 말아야 하는 요청에서 읽음": make_record(should_not_trigger, empty_evidence(), skill_reads=[SKILL_PATH]),
    "None: 사례에 skill_read 기대가 없음": make_record(no_trigger_expectation, empty_evidence(), skill_reads=[SKILL_PATH]),
}
show_check(trigger_ok, labelled)

**관찰할 것.** 넷째 줄의 실패는 "Skill을 읽은 것"이 실패입니다. 읽기가 늘 좋은 것이 아니라는 점이 Trigger Eval을 두 방향으로 만드는 이유입니다. 읽어야 하는 사례만 있으면 "항상 읽는 에이전트"가 만점을 받습니다.

### 읽었는데 결과가 나쁜 기록, 안 읽었는데 끝낸 기록

이제 같은 사례(dev-05, 모호한 완료 기준)에 대한 손으로 만든 기록 네 개를 만들어 두 질문을 나란히 봅니다. dev-05의 기대(`vague_answer`, `frozen`, `must_call`)에 이 노트북 안에서만 `skill_read: True`를 더한 사본을 씁니다. 근거 폴더는 서버의 업무 코드로 만들므로, "확정 명세의 완료 기준이 모호한 문장 그대로"인 상황이 파일에 정말로 남습니다.

- A: Skill을 읽었지만 사용자의 모호한 답을 **그대로** 확정했다.
- B: Skill을 읽지 않았지만 명확한 기준으로 확정까지 끝냈다. (이 저장소의 기본 구성에서는 MCP 도구가 Skill을 읽은 뒤에야 공개되므로 드문 경로입니다. 도구를 처음부터 공개하는 구성 `tool_disclosure="upfront"`(`cap_lab/agent.py`)에서는 가능합니다. 여기서는 점수가 어떻게 갈리는지만 봅니다.)
- C: Skill도 안 읽고 모호한 기준을 확정했다.
- D: Skill을 읽고 명확한 기준으로 확정했다.

**먼저 예측하세요.** A, B, C, D 각각의 `trigger_ok`와 `vague_not_frozen`을 먼저 적어 보세요. 그리고 `first_failure`가 C에서 어느 단계를 가리킬지도 적어 보세요.

In [ ]:
# [따라하기] Trigger와 Quality가 갈리는 네 기록 (dev-05의 사본에 skill_read 기대를 더했다)
dev_05 = next(case for case in dev_cases if case["id"] == "dev-05")
quality_case = {**dev_05, "expect": {**dev_05["expect"], "skill_read": True}}
print("사용하는 기대:", quality_case["expect"])

def full_flow_calls(criteria):
    """시작 -> 제약 -> 완료 기준 -> 확정까지 성공한 호출 네 개."""
    return [tool_call("start_interview", {"goal": "고객 문의를 유형별로 나눠 주는 도구"}),
            tool_call("record_answer", {"field": "constraints", "answer": "없음"}),
            tool_call("record_answer", {"field": "acceptance_criteria", "answer": criteria}),
            tool_call("freeze_seed", {})]


def finished_record(criteria, *, read_skill):
    """서버 코드로 인터뷰를 확정까지 진행한 근거 폴더와, 그에 맞는 손으로 만든 기록을 만든다."""
    workflow, evidence_dir = new_lab("quality")
    interview(workflow, "고객 문의를 유형별로 나눠 주는 도구", constraints="없음", criteria=criteria, freeze=True)
    return make_record(quality_case, evidence_dir, calls=full_flow_calls(criteria),
                       skill_reads=[SKILL_PATH] if read_skill else [])


four = {
    "A: 읽음 + 모호한 기준 확정": finished_record(VAGUE, read_skill=True),
    "B: 안 읽음 + 명확한 기준 확정": finished_record(CLEAR, read_skill=False),
    "C: 안 읽음 + 모호한 기준 확정": finished_record(VAGUE, read_skill=False),
    "D: 읽음 + 명확한 기준 확정": finished_record(CLEAR, read_skill=True),
}
rows = []
for label, record in four.items():
    scores = {score.name: score.value for score in score_run(record)}
    rows.append({"기록": label, "trigger_ok(S1)": scores["trigger_ok"],
                 "vague_not_frozen(S3)": scores["vague_not_frozen"], "frozen_ok(S5)": scores["frozen_ok"],
                 "first_failure": first_failure(score_run(record))})
show_table(rows)

**관찰할 것.** 표의 `NaN`은 `first_failure`가 `None`이라는 뜻입니다(D는 실패가 없음). A는 Trigger가 `True`인데 Quality(`vague_not_frozen`)가 `False`입니다. B는 반대로 Trigger가 `False`인데 Quality는 모두 `True`입니다. 두 점수를 하나로 합친 "좋은 실행인가"만 있었다면 A와 B는 둘 다 "나쁨"이 되어 **원인이 다른 두 실패가 한 덩어리**가 됩니다. A를 고치려면 Skill 본문(어떻게 모호함을 다루는지)이나 서버의 게이트를 봐야 하고, B를 고치려면 Skill의 `description`(언제 읽을지)을 봐야 합니다.

C 행의 `first_failure`도 보세요. S1이 먼저라서 `S1`만 나옵니다. C는 S3에서도 실패했지만 `first_failure`는 가장 이른 실패 하나만 돌려줍니다(`cap_lab/scorers.py:322`). **`first_failure`만 모아 보면 Quality 실패가 Trigger 실패 뒤에 숨습니다.** 단계별 통과율을 따로 내야 하는 이유입니다. 다음 셀이 그것을 보여 줍니다.

In [ ]:
# [바꿔보기] 합친 숫자와 단계별 숫자: 네 기록을 두 방식으로 보고한다
rows = [{"variant": "hand", **{score.name: score.value for score in score_run(record)}} for record in four.values()]
names = ["trigger_ok", "vague_not_frozen", "frozen_ok"]
show_table(pass_rate_table(rows, names))

all_good = sum(all(row[name] is True for name in names) for row in rows)
print(f"세 검사를 모두 통과한 기록: {all_good}/{len(rows)}  (합친 숫자는 어느 단계가 문제인지 말하지 않는다)")
print("첫 실패 단계 분포:", [first_failure(score_run(record)) for record in four.values()])

**관찰할 것.** 단계별 표에서는 `trigger_ok`가 2/4, `vague_not_frozen`이 2/4, `frozen_ok`가 4/4입니다. `frozen_ok`는 네 기록 모두 통과입니다. 모호한 기준이든 명확한 기준이든 **확정은 됐기 때문**입니다. 이것이 Quality Eval을 한 점수로 줄이면 안 되는 두 번째 이유입니다. "확정되었는가"(`frozen_ok`)는 통과하고 "확정된 내용이 좋은가"(`vague_not_frozen`)는 실패하는 기록이 있습니다. 합친 숫자(1/4)는 두 실패를 한 줄로 뭉개고, `first_failure` 분포는 S3 실패를 숨깁니다. 이 노트북에서 반복해서 말할 규칙입니다. **Trigger와 Quality는 따로 계산하고 따로 보고합니다.**

### frozen_ok와 vague_not_frozen의 None

`frozen_ok`(`cap_lab/scorers.py:209`)는 `expect.frozen`이 `True`면 확정 명세 파일이 **있어야** 통과하고, `False`면 **없어야** 통과합니다. dev-06처럼 "모호해서 확정하면 안 되는" 사례에서 확정하지 않는 것이 정답입니다. `None`은 기대가 없을 때입니다.

In [ ]:
# [따라하기] frozen_ok: 통과(확정됨), 통과(확정하지 않음이 정답), 실패, None
must_freeze = make_case("hand-S5-a", {"frozen": True})
must_not_freeze = make_case("hand-S5-b", {"frozen": False})
no_freeze_expectation = make_case("hand-S1-j", {"skill_read": True})

workflow_f, frozen_dir = new_lab("frozen")
interview(workflow_f, GOAL, constraints="없음", criteria=CLEAR, freeze=True)
workflow_u, unfrozen_dir = new_lab("unfrozen")
interview(workflow_u, GOAL, constraints="없음")  # 시작만 하고 완료 기준은 받지 못했다

labelled = {
    "통과: 확정해야 하는 사례에서 확정함": make_record(must_freeze, frozen_dir),
    "통과: 확정하면 안 되는 사례에서 확정하지 않음": make_record(must_not_freeze, unfrozen_dir),
    "실패: 확정해야 하는데 확정 파일이 없음": make_record(must_freeze, unfrozen_dir),
    "None: frozen 기대가 없음": make_record(no_freeze_expectation, frozen_dir),
}
show_check(frozen_ok, labelled)

### 두 번째 깨뜨리기: Trigger Eval은 경로의 끝부분만 본다

`trigger_ok`는 읽은 경로 문자열이 `requirements-interview/SKILL.md`로 **끝나는지**만 봅니다(`SKILL_SUFFIX`, `cap_lab/scorers.py:15`). 그렇다면 다른 폴더에 있는 같은 이름의 파일을 읽어도 "Skill을 읽음"이 될까요? 실제 에이전트는 Skill 폴더만 읽을 수 있게 막혀 있습니다(`cap_lab/agent.py`의 파일 도구 설정). 그래서 이 빈틈이 현재 구성에서는 실제로 일어나기 어렵습니다. 하지만 **점수 함수 자체의 성질**은 알아 둘 만합니다.

**먼저 예측하세요.** `skill_reads`에 `/tmp/남의-폴더/requirements-interview/SKILL.md`가 들어 있고 사례는 `skill_read: True`를 기대합니다. `trigger_ok`는 무엇을 돌려줄까요?

In [ ]:
# [깨뜨리기] 경로의 끝부분만 같으면 Trigger 통과가 되는가
should_read = make_case("hand-S1-k", {"skill_read": True})
fake_skill = "/tmp/남의-폴더/requirements-interview/SKILL.md"       # 같은 이름, 다른 폴더
other_file = "/requirements-interview/README.md"                   # 같은 폴더, 다른 파일

labelled = {
    "진짜 Skill 경로": make_record(should_read, empty_evidence(), skill_reads=[SKILL_PATH]),
    "다른 폴더의 같은 이름 파일": make_record(should_read, empty_evidence(), skill_reads=[fake_skill]),
    "같은 폴더의 다른 파일": make_record(should_read, empty_evidence(), skill_reads=[other_file]),
}
show_check(trigger_ok, labelled)

**관찰할 것.** 둘째 줄이 `True`입니다. 점수 함수는 파일 내용도, 폴더가 어디인지도 확인하지 않고 끝부분의 문자열만 봅니다. 이것은 버그라기보다 **측정이 무엇을 보증하는지의 경계**입니다. "Skill을 읽었다"는 주장의 근거는 (1) 에이전트가 읽을 수 있는 폴더를 Skill 폴더로 제한한 구성과 (2) 경로 끝부분 비교, 두 가지가 합쳐진 것입니다. 하나만 보면 부족합니다. 평가 결과를 보고할 때 "무엇을 세었는가"를 말해야 하는 이유입니다. 이 경계는 원문(`cap_lab/scorers.py:123-131`)을 읽지 않으면 알 수 없습니다.

## 3. 권한 거부 · 오류 · 재시도 · 중복 실행

운영 환경의 에이전트는 도구가 늘 성공한다는 가정을 하면 안 됩니다. 이 저장소는 서버가 장애를 **일부러 주입**하는 옵션을 가집니다(`cap_lab/server/server.py:64`의 `FaultPlan`). 사례가 `fault`를 가지면 실행기가 서버를 그 설정으로 띄웁니다.

| 장애 | 서버 동작 | 올바른 에이전트 | 검사 |
|---|---|---|---|
| `permission_denied: [도구]` | 그 도구의 **모든** 호출을 `permission_denied`로 거절 | 같은 호출을 되풀이하지 않고 사용자에게 알린다 | R1 `no_blind_retry` |
| `transient: {도구: n}` | 그 도구의 처음 n번을 `temporarily_unavailable`(`retryable`)로 거절하고 이후 정상 | 제한된 횟수 안에서 다시 시도해 성공시킨다 | R2 `retry_bounded` |
| (장애 없음) 같은 요청을 두 번 | 서버가 중복을 흡수하는지는 도구마다 다르다 | 같은 요청이 상태를 두 번 바꾸지 않는다 | R3 `side_effect_stable` |

장애 주입은 도구에 닿기 **전에** 결과를 돌려줍니다(`cap_lab/server/server.py:83`의 docstring). 업무 코드가 실행되지 않으므로 장애가 난 호출은 근거에도 흔적을 남기지 않습니다. 실제로 그런지 서버에 직접 확인해 봅니다. 아래 셀은 `record_answer`의 첫 호출만 일시 오류로, `save_evidence`는 전부 거절로 설정한 서버에 호출을 보냅니다.

**먼저 예측하세요.** `record_answer`를 같은 인자로 두 번, `save_evidence`를 같은 인자로 두 번 부르면 각 호출의 `success`와 `error`는 무엇일까요? 그리고 근거 폴더에는 이벤트가 몇 개 남을까요?

In [ ]:
# [따라하기] 실제 서버의 장애 주입: 일시 오류는 한 번 실패하고, 권한 거부는 계속 실패한다 (모델 없음)
fault_plan = FaultPlan(permission_denied=["save_evidence"], transient={"record_answer": 1})
fault_workdir = Path(tempfile.mkdtemp(prefix="lab06-fault-"))

attempts = []
async with connect_server(fault_workdir, fault=fault_plan) as client:
    started = await client.call_tool("start_interview", {"goal": GOAL})
    fault_session = started.structured_content["session_id"]
    answer_args = {"session_id": fault_session, "field": "constraints", "answer": "없음"}
    save_args = {"session_id": fault_session, "title": "요약", "content": "본문"}
    for label, tool, arguments in [
        ("record_answer 1번째", "record_answer", answer_args),
        ("record_answer 2번째(같은 인자)", "record_answer", answer_args),
        ("save_evidence 1번째", "save_evidence", save_args),
        ("save_evidence 2번째(같은 인자)", "save_evidence", save_args),
    ]:
        outcome = (await client.call_tool(tool, arguments)).structured_content
        attempts.append({"호출": label, "success": outcome.get("success"), "error": outcome.get("error"),
                         "retryable": outcome.get("retryable")})

show_table(attempts)
print("근거 폴더의 이벤트:")
show_events(fault_workdir / "evidence")

**관찰할 것.** `record_answer`는 첫 호출이 `temporarily_unavailable`(`retryable=True`), 둘째가 성공입니다. `save_evidence`는 몇 번을 불러도 `permission_denied`이고 `retryable` 칸이 비어 있습니다. 이 한 칸이 에이전트에게 주는 **재시도해도 되는가**의 신호입니다. 근거의 이벤트는 `interview_started` 하나와 `answer_recorded` 하나뿐입니다. 실패한 호출은 상태를 바꾸지 않았고, 성공한 호출만 이벤트를 남겼습니다. 재시도가 안전한 이유가 여기에 있습니다.

### R1: no_blind_retry (권한 거부 뒤 같은 호출을 되풀이하지 않는다)

`no_blind_retry`(`cap_lab/scorers.py:240`)는 `fault.permission_denied`가 있는 사례에서만 켜집니다. 거절된 도구마다 **같은 인자(JSON 직렬화가 같은 호출)** 가 몇 번 불렸는지 세어서, 사례의 `max_same_call`(기본 1)을 넘으면 실패입니다. 실제 dev-09를 씁니다.

In [ ]:
# [따라하기] no_blind_retry: dev-09(save_evidence 권한 거부, 같은 호출은 1번까지)의 통과, 실패, None
dev_09 = next(case for case in dev_cases if case["id"] == "dev-09")
dev_01 = next(case for case in dev_cases if case["id"] == "dev-01")
print("dev-09 fault:", dev_09["fault"], "/ expect:", dev_09["expect"])

denied = {"success": False, "error": "permission_denied", "message": "이 도구를 사용할 권한이 없습니다."}
same_save = {"session_id": "b" * 32, "title": "요약", "content": "최종 요약 본문"}
labelled = {
    "통과: 거절된 save_evidence를 한 번만 부르고 멈춤": make_record(dev_09, empty_evidence(), calls=[
        tool_call("save_evidence", same_save, result=denied, ok=False)]),
    "실패: 같은 인자로 3번 되풀이함": make_record(dev_09, empty_evidence(), calls=[
        tool_call("save_evidence", same_save, result=denied, ok=False) for _ in range(3)]),
    "None: 권한 거부 장애가 없는 사례(dev-01)": make_record(dev_01, empty_evidence(), calls=[
        tool_call("save_evidence", same_save, result=denied, ok=False) for _ in range(3)]),
}
show_check(no_blind_retry, labelled)

**관찰할 것.** 셋째 줄은 같은 세 번 호출인데 `None`입니다. dev-01에는 권한 거부가 설정되어 있지 않아서 이 검사가 "해당 없음"이기 때문입니다. 검사는 **호출 기록만 보고 켜지지 않고 사례의 장애 설정이 켭니다.** 이것도 실행을 읽을 때 놓치기 쉬운 부분입니다. 그리고 첫 번째 통과 기록의 "멈춤"은 정확히는 "같은 호출을 `max_same_call`번 넘게 하지 않음"입니다. 멈추고 사용자에게 알렸는지는 이 검사가 보지 않습니다.

### 세 번째 깨뜨리기: 인자를 조금 바꿔 계속 재시도하면?

`no_blind_retry`는 인자가 **완전히 같은** 호출만 셉니다. 거절된 도구를 인자를 살짝 바꿔 계속 부르면 어떻게 될까요? 모델이 실제로 그렇게 행동할 수 있습니다. 거절 메시지를 보고 "제목이 문제인가?" 하고 제목을 바꿔서 다시 보내는 식입니다.

**먼저 예측하세요.** 같은 인자 3번(실패)과, 제목에 `(재시도 1)`, `(재시도 2)`를 붙여 총 5번 부른 기록 중 어느 쪽이 `no_blind_retry`를 통과할까요? 도구 호출 수(`tool_calls`)는 어느 쪽이 클까요?

In [ ]:
# [깨뜨리기] 영구적인 권한 거부 아래의 재시도 루프: 인자를 바꾸면 no_blind_retry를 빠져나간다
def retry_calls(titles):
    """제목만 바꿔 save_evidence를 부른 호출 기록들 (모두 권한 거부로 돌아온다)."""
    return [tool_call("save_evidence", {**same_save, "title": title}, result=denied, ok=False) for title in titles]


identical_loop = make_record(dev_09, empty_evidence(), calls=retry_calls(["요약"] * 3))
varied_loop = make_record(dev_09, empty_evidence(),
                          calls=retry_calls(["요약"] + [f"요약 (재시도 {n})" for n in range(1, 5)]))

labelled = {"같은 인자 3번": identical_loop, "제목을 바꿔 5번": varied_loop}
show_check(no_blind_retry, labelled)
for label, record in labelled.items():
    print(f"{label}: tool_calls = {int(next(s.value for s in score_run(record) if s.name == 'tool_calls'))}")

**관찰할 것.** 5번 부른 기록이 `no_blind_retry`를 통과합니다. 호출은 더 많은데 점수는 더 좋습니다. 점수 함수가 "같은 인자"라는 기준을 썼기 때문에, 의도는 같은 무익한 재시도를 인자의 사소한 변화로 우회한 것을 알아보지 못합니다. 이 빈틈을 메우려면 두 가지 방법이 있습니다. 도구별 총 호출 수 상한을 두는 방법(다만 정상적인 재시도와 구별하기 어렵습니다)이 하나이고, 거절 코드(`permission_denied`) 뒤에는 같은 도구를 부르지 않는다는 규칙으로 바꾸는 방법이 다른 하나입니다. 어느 쪽이든 **사전 등록된 점수 함수를 결과를 본 뒤에 바꾸지 않습니다.** 이 노트북에서는 빈틈을 발견했다는 사실을 기록하는 것으로 충분합니다. 그리고 숫자 칸의 `tool_calls`는 이 우회를 보여 주는 가드 지표입니다. 통과율 옆에 늘 같이 보세요.

### R2: retry_bounded (일시 오류 뒤 재시도가 상한 안이고 답이 기록되었다)

`retry_bounded`(`cap_lab/scorers.py:252`)는 `fault.transient`가 있는 사례에서 두 가지를 봅니다. (1) 장애가 설정된 도구의 호출 수가 `장애 횟수 + 2` 이하인가. dev-10은 `record_answer`가 1번 실패하도록 설정되어 있으므로 상한이 3입니다. (2) 근거 폴더에 `answer_recorded` 이벤트가 **있는가.** 호출 수만 보면 "한 번 시도하고 포기"도 통과가 되기 때문입니다. 호출 수가 상한 안이어도 이벤트가 없으면 실패입니다.

In [ ]:
# [따라하기] retry_bounded: dev-10(record_answer 1번 일시 오류)의 통과, 실패 2종, None
dev_10 = next(case for case in dev_cases if case["id"] == "dev-10")
print("dev-10 fault:", dev_10["fault"], "/ expect:", dev_10["expect"])

transient = {"success": False, "error": "temporarily_unavailable", "retryable": True}
answer = {"session_id": "c" * 32, "field": "constraints", "answer": "운전자 개인 연락처 비노출"}
fail_then_ok = [tool_call("record_answer", answer, result=transient, ok=False), tool_call("record_answer", answer)]

workflow_r, recorded_dir = new_lab("recorded")
interview(workflow_r, "차량 배차 도구", constraints="운전자 개인 연락처 비노출")      # answer_recorded 이벤트가 있다
workflow_n, started_only_dir = new_lab("started-only")
interview(workflow_n, "차량 배차 도구")                                             # 시작만 했다

labelled = {
    "통과: 1번 실패 뒤 다시 불러 기록됨(호출 2번)": make_record(dev_10, recorded_dir, calls=fail_then_ok),
    "실패: 호출 5번(상한 3 초과)": make_record(dev_10, recorded_dir, calls=fail_then_ok + fail_then_ok + fail_then_ok[:1]),
    "실패: 호출 1번에서 포기(이벤트 없음)": make_record(dev_10, started_only_dir, calls=fail_then_ok[:1]),
    "None: 일시 오류 장애가 없는 사례(dev-09)": make_record(dev_09, recorded_dir, calls=fail_then_ok),
}
show_check(retry_bounded, labelled)

다음은 경계 실험입니다. 같은 근거 폴더(`answer_recorded`가 있음)로 `record_answer` 호출 수만 1번에서 5번까지 바꿔 보면 경계는 어디일까요? 상한은 `장애 횟수 + 2`입니다. 이것은 값을 올려서 실패를 감추는 실험이 아닙니다. **점수 함수의 경계 위치를 이해하는 실험**이고, 사례 파일의 값은 바꾸지 않습니다.

In [ ]:
# [바꿔보기] 호출 수에 따라 retry_bounded가 어디서 바뀌는가 (상한 = 장애 횟수 + 2)
failures = dev_10["fault"]["transient"]["record_answer"]
rows = []
for number_of_calls in range(1, 6):
    calls = [tool_call("record_answer", answer, result=transient, ok=False)] + \
            [tool_call("record_answer", answer) for _ in range(number_of_calls - 1)]
    score = retry_bounded(make_record(dev_10, recorded_dir, calls=calls))
    rows.append({"record_answer 호출 수": number_of_calls, "상한": failures + 2, "값": score.value,
                 "이유(comment)": score.comment})
show_table(rows)

**관찰할 것.** 호출 3번까지 `True`, 4번부터 `False`입니다. 상한 안의 호출이 모두 합리적이라는 뜻은 아닙니다. 1번 실패 뒤 성공한 호출을 2번 더 부르는 것은 낭비이지만 이 검사는 통과시킵니다. 이 검사가 답하는 질문은 "재시도가 **폭주하지 않았는가**"이지 "재시도가 **효율적이었는가**"가 아닙니다. 효율은 `tool_calls`와 `latency_s` 같은 숫자 칸이 보여 줍니다.

### R3: side_effect_stable (중복 실행이 상태를 늘리지 않는다)

`side_effect_stable`(`cap_lab/scorers.py:270`)은 근거 폴더의 모든 세션에서 `interview_started`, `answer_recorded`, `seed_frozen` 이벤트의 **총 개수**가 `expect.event_count`와 같은지 봅니다. dev-07(전체 흐름)의 기대는 4입니다. 시작 1, 답 2(제약, 완료 기준), 확정 1입니다.

서버는 같은 답변을 다시 받으면 이벤트를 늘리지 않습니다(`cap_lab/server/workflow.py:204`). 그러나 `start_interview`를 한 번 더 부르면 **새 세션**이 만들어져 이벤트가 하나 늘어납니다(`cap_lab/server/workflow.py:171`, 호출마다 새 `uuid4`). 두 경우를 직접 만들어 봅니다.

In [ ]:
# [따라하기] side_effect_stable: dev-07(이벤트 4개 기대)의 통과, 실패(세션 중복), None
dev_07 = next(case for case in dev_cases if case["id"] == "dev-07")
print("dev-07 event_count 기대:", dev_07["expect"]["event_count"])

workflow_ok, stable_dir = new_lab("stable")
sid = interview(workflow_ok, "출장 영수증을 모아 월별 정산표를 만드는 도구", constraints="외부 전송 없음", criteria=CLEAR)
workflow_ok.record_answer(sid, "constraints", "외부 전송 없음")  # 같은 답을 다시 보낸다 (서버가 흡수)
workflow_ok.freeze_seed(sid)                                     # 확정 뒤에는 답을 바꿀 수 없으므로 그 전에 보냈다

workflow_dup, duplicated_dir = new_lab("duplicated")
interview(workflow_dup, "출장 영수증을 모아 월별 정산표를 만드는 도구", constraints="외부 전송 없음", criteria=CLEAR, freeze=True)
workflow_dup.start_interview("출장 영수증을 모아 월별 정산표를 만드는 도구")  # 요청 하나에 세션을 또 만들었다

labelled = {
    "통과: 같은 답을 두 번 보냈지만 이벤트는 4개": make_record(dev_07, stable_dir),
    "실패: start_interview를 두 번 불러 세션이 둘": make_record(dev_07, duplicated_dir),
    "None: event_count 기대가 없는 사례(dev-01)": make_record(dev_01, duplicated_dir),
}
show_check(side_effect_stable, labelled)

**관찰할 것.** 같은 답을 두 번 보낸 기록은 통과, 새 세션을 또 만든 기록은 5개라서 실패입니다. 중복 호출의 **서버 쪽 처리**가 도구마다 다르다는 것이 요점입니다. `record_answer`는 같은 답이면 아무것도 쓰지 않고(멱등), `start_interview`는 호출마다 새 세션을 만듭니다. 이 때문에 Skill 본문에는 "요청 하나에 세션은 하나다. `start_interview`가 세션 ID를 돌려준 뒤에는 같은 요청으로 다시 호출하지 않는다"라는 문장이 있습니다(`skills/requirements-interview/SKILL.md`). 서버가 막을 수 없는 중복은 지시문이 막고, 그 지시문이 지켜졌는지는 이 검사가 확인합니다.

### contamination_free: 평가 자료를 건드리지 않았는가

마지막 검사입니다. `contamination_free`(`cap_lab/scorers.py:285`)는 도구 인자(JSON으로 직렬화한 전체)와 읽은 Skill 경로에 평가 자료를 가리키는 표지(`CONTAMINATION_MARKERS`, `cap_lab/scorers.py:22`) 중 하나가 들어 있으면 실패입니다. 표지의 실제 값은 바로 아래 셀이 출력합니다. 에이전트가 평가 사례 파일에서 답을 훔쳐보지 못하게 하는 장치입니다. 모든 실행에 적용되어서 사례에 상관없이 `True`나 `False`이고, 실행 오류가 있으면 `score_run`이 통과를 `None`으로 돌립니다. 오염된 사실(`False`)은 오류가 있어도 지우지 않습니다(`cap_lab/scorers.py:299-318`). 실패를 숨기는 방향으로는 절대 바뀌지 않게 한 것입니다.

한 검사의 세 값을 `score_run`으로 확인합니다.

In [ ]:
# [읽기] 오염 표지: 도구 인자나 읽은 Skill 경로에 이 문자열이 있으면 contamination_free가 실패한다
from cap_lab.scorers import CONTAMINATION_MARKERS

print("평가 자료 오염 표지:", CONTAMINATION_MARKERS)


In [ ]:
# [따라하기] contamination_free: 통과, 실패, None(실행 오류가 있으면 통과를 확인 불가로 돌린다)
clean_case = make_case("hand-all", {})
leak_args = {"session_id": "d" * 32, "title": "참고", "content": "Day04_new/data/cases.jsonl 의 정답을 보고 작성"}

clean = make_record(clean_case, empty_evidence(), calls=[start_call])
leaked = make_record(clean_case, empty_evidence(), calls=[tool_call("save_evidence", leak_args)])
errored_clean = make_record(clean_case, empty_evidence(), calls=[start_call], error="TimeoutError: 시간 초과")
errored_leaked = make_record(clean_case, empty_evidence(), calls=[tool_call("save_evidence", leak_args)],
                             error="TimeoutError: 시간 초과")

rows = []
for label, record in {"통과: 깨끗한 실행": clean, "실패: 도구 인자에 cases.jsonl": leaked,
                      "None: 깨끗하지만 실행 오류": errored_clean,
                      "실패 유지: 오염 + 실행 오류": errored_leaked}.items():
    score = next(s for s in score_run(record) if s.name == "contamination_free")
    rows.append({"손으로 만든 기록": label, "값": score.value, "이유(comment)": score.comment})
show_table(rows)

**관찰할 것.** 셋째 줄이 `None`, 넷째 줄은 오류가 있어도 `False`입니다. "오류로 확인 불가"가 오염 사실을 덮지 않는다는 비대칭이 코드에 의도적으로 들어 있습니다. 불확실할 때 좋은 쪽이 아니라 나쁜 쪽을 남기는 방향입니다.

### 네 번째 깨뜨리기: 표지가 너무 넓으면

`/data/`는 평가 폴더를 가리키는 표지로 넣었지만 문자열 비교는 의미를 모릅니다. 평가 자료와 아무 상관 없는 인자도 걸릴 수 있습니다.

**먼저 예측하세요.** `save_evidence`의 본문이 "고객이 올린 파일은 s3://bucket/data/export 에 둔다"라는 정상적인 요구사항 문장이라면 `contamination_free`는 무엇을 돌려줄까요? 에이전트는 평가 자료를 건드리지 않았는데도 말입니다.

In [ ]:
# [깨뜨리기] 평가 자료와 무관한 정상 문장이 오염으로 판정되는가 (거짓 양성)
innocent_args = {"session_id": "e" * 32, "title": "저장 위치 요구사항",
                 "content": "고객이 올린 파일은 s3://bucket/data/export 에 둔다."}
innocent = make_record(clean_case, empty_evidence(), calls=[tool_call("save_evidence", innocent_args)])
show_check(contamination_free, {"정상 문장(/data/ 포함)": innocent, "평가 파일을 직접 가리킴": leaked})

**관찰할 것.** 정상 문장도 `False`입니다. 점수 함수는 표지 문자열을 찾을 뿐, 에이전트가 그 경로를 **읽었는지**는 모릅니다. 이 거짓 양성은 보수적인 선택입니다. 오염을 놓치는 것보다 정상을 오염으로 의심하는 쪽이 평가의 신뢰에는 안전합니다. 대신 이 값이 `False`로 나왔을 때 바로 "오염"이라고 결론 내리지 말고 **어떤 표지가 어느 인자에서 걸렸는지 comment를 읽고 원문으로 확인**해야 합니다. 이유 문장에는 걸린 표지(`/data/`)가 적혀 있습니다. 이것이 다음 4절의 디버깅 절차입니다.

## 4. 디버깅 걷기: 실패한 점수에서 호출과 근거 파일까지

점수가 `False`일 때의 절차는 늘 같습니다.

1. **점수표에서 `False`인 칸을 찾는다.** `None`은 건너뛴다(조사할 대상이 아니라 확인되지 않은 대상이다).
2. **`first_failure`로 어느 단계에서 처음 틀렸는지 본다.** 가장 이른 단계(S1, S2, S3, S5, S6 순)가 먼저 고칠 곳이다. R1~R3은 이 순서에 들어가지 않고 따로 읽는다.
3. **그 점수의 `comment`를 읽는다.** 점수 함수가 무엇을 세다가 틀렸다고 판단했는지 문장으로 적혀 있다.
4. **그 점수와 관련된 도구 호출을 본다.** 어떤 인자로 불렀고 결과가 무엇이었는지.
5. **근거 폴더의 파일을 열어 서버가 실제로 쓴 내용을 확인한다.** 호출 기록은 에이전트가 한 일이고, 근거 파일은 서버가 받은 일이다. 둘이 다르면 거기가 원인이다.

아래 셀은 이 절차를 함수로 묶습니다. `RELATED_TOOLS`는 점수마다 어느 도구 호출이 관련되는지를 정한 표입니다. 이 표는 이 노트북이 편의로 정한 것이지 `cap_lab`의 일부가 아닙니다. 점수 함수는 관련 호출을 따로 돌려주지 않으므로, 각 함수가 읽는 필드(`cap_lab/scorers.py`)에서 정리했습니다.

In [ ]:
# [따라하기] 디버깅 도구: 실패한 점수 -> 관련 호출 -> 근거 파일
def related_tools(score_name, record):
    """점수 하나와 관련된 도구 이름들. 각 점수 함수가 읽는 호출에서 정리한 표다(이 노트북의 편의)."""
    expect, fault = record.case["expect"], record.case.get("fault") or {}
    table = {
        "trigger_ok": {"read_file"},
        "tool_selection_ok": set(expect.get("must_call") or []) | set(expect.get("must_not_call") or []),
        "args_valid": set(MCP_TOOLS),
        "goal_preserved": {"start_interview"},
        "vague_not_frozen": {"record_answer", "freeze_seed"},
        "frozen_ok": {"record_answer", "freeze_seed"},
        "evidence_linked": {"save_evidence", "read_evidence"},
        "no_blind_retry": set(fault.get("permission_denied") or []),
        "retry_bounded": set((fault.get("transient") or {}).keys()),
        "side_effect_stable": {"start_interview", "record_answer", "freeze_seed"},
        "contamination_free": {call["name"] for call in record.tool_calls} | {"read_file"},
    }
    return table.get(score_name, set())


def summarize_result(result):
    """도구 결과를 한 줄로: 업무 결과는 success와 error 코드, 텍스트 블록은 앞부분."""
    if isinstance(result, dict):
        return f"success={result.get('success')} error={result.get('error')}"
    if isinstance(result, list) and result and isinstance(result[0], dict):
        return str(result[0].get("text", result[0]))[:70].replace("\n", " ")
    return str(result)[:70].replace("\n", " ")


def show_tool_calls(record, only=None):
    """도구 호출을 순서대로 표로. only에 이름 집합을 주면 그 도구만 보여 주되 원래 순번은 유지한다."""
    rows = [{"순번": number, "도구": call["name"], "인자": json.dumps(call["args"], ensure_ascii=False)[:90],
             "ok": call["ok"], "결과 요약": summarize_result(call["result"])}
            for number, call in enumerate(record.tool_calls, start=1) if only is None or call["name"] in only]
    show_table(rows) if rows else print("(해당하는 호출 없음)")


def show_evidence_files(record):
    """근거 폴더의 파일을 세션별로 나열한다. 확정 명세(seed)는 내용도 보여 준다."""
    paths = sorted(Path(record.evidence_dir).glob("*/*.json")) if Path(record.evidence_dir).is_dir() else []
    for path in paths:
        saved = json.loads(path.read_text(encoding="utf-8"))
        extra = f" -> {saved['content']}" if saved["kind"] == "seed" else ""
        print(f"  {path.parent.name[:8]}/{path.name} [{saved['kind']}]{extra}")
    if not paths:
        print("  (근거 파일 없음)")


def trace_failure(record):
    """실패한 점수마다 단계, 이유, 관련 호출, 근거 파일을 차례로 보여 준다."""
    scores = score_run(record)
    failed = [score for score in scores if score.value is False]
    print(f"== 사례 {record.case['id']} / first_failure = {first_failure(scores)} / 오류 = {record.error}")
    if not failed:
        print("실패한 검사가 없습니다. (None인 검사:", [s.name for s in scores if s.value is None and s.name in STAGE_OF], ")")
        return
    for score in failed:
        print(f"\n[{score.name}] 단계 {STAGE_OF.get(score.name, '전체')} -> {score.comment}")
        show_tool_calls(record, only=related_tools(score.name, record))
    print("\n근거 파일:")
    show_evidence_files(record)


print("디버깅 도구 준비 완료")

먼저 손으로 만든 기록으로 걸어 봅니다. 앞에서 만든 기록 A(Skill을 읽었지만 모호한 기준을 확정함)입니다. 호출 기록만 보면 `freeze_seed`까지 모두 성공했습니다. 어디가 문제인지는 근거 파일에서 드러납니다.

**먼저 예측하세요.** `trace_failure`가 가리킬 단계는 무엇이고, 근거 파일의 어느 줄에서 모호한 문장을 볼 수 있을까요?

In [ ]:
# [따라하기] 디버깅 걷기 1: 모호한 기준이 확정된 기록 A
trace_failure(four["A: 읽음 + 모호한 기준 확정"])

**관찰할 것.** 실패한 점수는 `vague_not_frozen` 하나이고 단계는 S3입니다. 이유 문장이 "모호한 답이 그대로 확정 명세가 됨"이라고 말합니다. 관련 호출을 보면 `freeze_seed`가 정상적으로 호출되었습니다(`ok=True`). 이 기록에서 **호출은 모두 성공했습니다.** 인자 칸에 모호한 문장이 보이긴 하지만 호출이 실패하지는 않았습니다. 서버가 실제로 확정한 내용은 근거 파일 목록의 `seed-...` 줄에서 확인합니다. 확정 명세의 `acceptance_criteria`가 사용자의 모호한 문장 그대로입니다. 호출은 성공했고 내용이 나빴습니다. 에이전트가 한 일(호출 기록)과 서버가 받은 일(근거 파일)을 둘 다 봐야 하는 이유입니다. 이제 R1 실패를 같은 방식으로 걸어 봅니다.

In [ ]:
# [따라하기] 디버깅 걷기 2: 권한 거부 뒤 같은 호출을 되풀이한 기록 (세션은 서버 코드로 만든 진짜 세션)
workflow_w, walk_dir = new_lab("walk")
walk_session = interview(workflow_w, "사내 교육 신청을 접수하는 도구")
walk_save = {"session_id": walk_session, "title": "요약", "content": "최종 요약 본문"}
trace_failure(make_record(dev_09, walk_dir, calls=[
    tool_call("start_interview", {"goal": "사내 교육 신청을 접수하는 도구"}),
    *[tool_call("save_evidence", walk_save, result=denied, ok=False) for _ in range(3)]]))

**관찰할 것.** 이번에는 단계가 `None`입니다. `first_failure`는 S1~S6만 훑고 R 단계(R1)는 포함하지 않기 때문입니다(`cap_lab/scorers.py:26`의 `STAGE_ORDER`). 실패가 있는데 `first_failure`가 `None`이라는 점을 놓치면 "실패 없음"으로 읽게 됩니다. 이유 문장에는 "거부된 save_evidence을 같은 인자로 3번 호출(허용 1)"이 있고, 관련 호출 표에 세 번이 모두 `error=permission_denied`로 보입니다. 이 호출들은 근거 폴더에 흔적이 없습니다(서버가 도구에 닿기 전에 거절했으므로). 근거 파일 목록에 `event-00000001.json`(세션 시작)만 있고 산출물(`artifact-...`) 파일이 없는 것이 바로 단서입니다. 세 번 시도했지만 서버는 한 번도 저장하지 않았습니다.

## 5. 실제 실행 5회: 기준선 Deep Agent를 dev 사례로 돌린다

지금까지는 손으로 만든 기록이었습니다. 이제 진짜 Deep Agent(Claude `claude-sonnet-5-5`, effort medium)를 dev 사례 5개로 한 번씩 돌립니다. 게이트를 모두 끈 **기준선**(`Variant(name="baseline")`, `cap_lab/experiment.py:66`)입니다. 사례는 다음과 같습니다.

| 사례 | 단계 | 무엇을 보는가 |
|---|---|---|
| dev-01 | S1 | Skill을 읽어야 하는 요청 (`trigger_ok`, `tool_selection_ok`) |
| dev-02 | S1 | Skill을 쓰지 말아야 하는 요청 (개념 질문) |
| dev-03 | S2 | 사용자의 목표 문장을 한 글자도 바꾸지 않고 `goal`에 넣는가 (`goal_preserved`) |
| dev-09 | R1 | `save_evidence`가 권한 거부될 때 되풀이하지 않는가 |
| dev-10 | R2 | `record_answer`가 한 번 일시 오류일 때 제한된 재시도로 성공하는가 |

`run_case`(`cap_lab/experiment.py:131`)는 사례마다 새 DB와 새 근거 폴더(`workdir/baseline-r1/<사례 ID>/`)에서 실행하고, 예외를 밖으로 내지 않고 `RunRecord.error`에 남깁니다. 에이전트는 사례의 `turns`(사용자 발화)만 받고 `expect`는 점수 계산에만 씁니다. Langfuse에는 연결하지 않습니다.

**1회 관찰입니다.** 같은 사례를 한 번만 돌리므로 여기서 나온 통과나 실패는 확정적 결론이 아닙니다. 모델은 같은 입력에도 다르게 행동할 수 있습니다. 그 차이를 재려면 같은 사례를 3회 이상 반복해야 하고(직접 해보기의 `RUN_OPTIONAL`), 변형을 비교하려면 07에서 하는 반복 설계가 필요합니다. 오늘은 "점수표를 읽는 법"을 익히는 용도입니다.

**먼저 예측하세요.** 다섯 사례 중 어느 것이 `False`를 받을 것 같은가요? 어느 검사에서 `None`이 나올지도 적어 보세요. (힌트: 0절 표의 "적용 조건"을 보면 사례마다 켜지는 검사를 알 수 있습니다.)

In [ ]:
# [따라하기] 기준선 에이전트로 dev 사례 5개를 한 번씩 실행한다 (유료 호출)
live_case_ids = ["dev-01", "dev-02", "dev-03", "dev-09", "dev-10"]
announce("기준선 dev 사례 5개 실행", model_calls=45, runs=5)  # 사례당 Claude 5~9회의 상한 x 5

live_settings = load_settings()          # Day04_new/.env 한 파일에서 읽는다
baseline = Variant(name="baseline")      # 게이트를 모두 끈 기준선
live_workdir = new_workdir("lab06-live")
cases_by_id = {case["id"]: case for case in dev_cases}

live_records = {}
for case_id in live_case_ids:
    record = await run_case(cases_by_id[case_id], variant=baseline, settings=live_settings,
                            workdir=live_workdir, repeat=1)
    live_records[case_id] = record
    print(f"{case_id}: 모델 호출 {record.model_calls}회, 도구 호출 {len(record.tool_calls)}회, "
          f"{record.latency_s:.0f}초, 오류 = {record.error}")
print("기록 위치:", live_workdir)

**관찰할 것.** 각 사례의 `오류` 칸을 먼저 보세요. `None`이 아니면 그 실행은 중간에 멈춘 것이라서 점수 함수가 적용 가능한 검사를 모두 `None`(확인 불가)으로 돌립니다(`cap_lab/scorers.py:299`). 오류가 난 실행을 "실패"라고 읽으면 안 됩니다. 도구 호출 수는 사례마다 크게 다릅니다. dev-02(개념 질문)는 거의 0회이고 dev-09와 dev-10은 서버 장애 때문에 호출이 늘어날 수 있습니다.

### 점수표 읽기

`score_run`으로 점수를 계산하고 `show_table`로 보여 줍니다. 11개 검사와 `first_failure`, 그리고 비용 숫자 두 개(`model_calls`, `tool_calls`)를 한 표에 놓습니다.

In [ ]:
# [따라하기] 실제 실행 5개의 점수표
if "live_records" not in globals():
    raise RuntimeError("위의 실제 실행 셀을 먼저 실행하세요.")

live_scores = {case_id: score_run(record) for case_id, record in live_records.items()}
score_rows = []
for case_id, scores in live_scores.items():
    by_name = {score.name: score.value for score in scores}
    score_rows.append({"사례": case_id, **{name: by_name[name] for name in CHECK_NAMES},
                       "first_failure": first_failure(scores),
                       "model_calls": by_name["model_calls"], "tool_calls": by_name["tool_calls"]})
show_table(score_rows)

**관찰할 것.** 한 행을 가로로 읽으면서 `None`과 `False`를 구분하세요. 표의 칸이 `None`이면 그 사례가 그 검사를 요구하지 않은 것(또는 실행 오류)입니다. `False`만 조사 대상입니다. 5개 실행에서 한 번씩만 나온 값이므로 "기준선은 dev-03에서 목표를 바꾼다" 같은 일반화는 할 수 없습니다. 말할 수 있는 것은 **"이 한 번의 실행에서 이 점수가 이 이유로 나왔다"** 까지입니다.

### R1과 R2를 근거와 대조하기

dev-09와 dev-10은 서버가 장애를 주입한 실행입니다. 점수가 말하는 내용이 근거 폴더와 맞는지 확인합니다. 각 사례의 도구 호출 전체와 서버가 쓴 이벤트(`show_events`)를 나란히 봅니다.

**먼저 예측하세요.** dev-09에서 `save_evidence`가 몇 번 불렸을까요? 근거 폴더에는 이벤트가 몇 개 있을까요(`save_evidence`는 이벤트가 아니라 산출물 파일을 남깁니다)? dev-10에서 `record_answer`가 `answer_recorded` 이벤트를 만들려면 몇 번 불려야 할까요?

In [ ]:
# [따라하기] R1(dev-09)과 R2(dev-10): 호출 기록과 서버가 쓴 근거 이벤트를 나란히 본다
for case_id in ("dev-09", "dev-10"):
    record = live_records[case_id]
    scores = {score.name: score for score in live_scores[case_id]}
    print(f"===== {case_id} fault = {record.case['fault']}")
    for name in ("no_blind_retry", "retry_bounded", "frozen_ok", "side_effect_stable"):
        if scores[name].value is not None:
            print(f"  {name} = {scores[name].value}: {scores[name].comment}")
    print("  -- 도구 호출 전체")
    show_tool_calls(record)
    print("  -- 서버가 근거 폴더에 쓴 이벤트")
    show_events(record.evidence_dir)

**관찰할 것.** 호출 기록에서 `error=temporarily_unavailable` 또는 `error=permission_denied`인 줄 바로 다음 줄을 보세요. 에이전트가 그 오류에 어떻게 반응했는지가 거기 있습니다. dev-10에서 `answer_recorded` 이벤트가 있다면 재시도가 성공한 것이고 없다면 중간에 포기한 것입니다. 같은 점수 `False`라도 호출이 상한을 넘은 경우와 이벤트가 없는 경우는 다른 실패입니다. comment가 둘을 구별해서 적어 줍니다. 서버 장애로 `tool_calls`가 늘었을 때 그것이 정상적인 재시도인지 폭주인지는 숫자만으로는 알 수 없고, 어느 호출이 어느 결과로 돌아왔는지 순서대로 읽어야 합니다.

### 실제 실행에서의 Skill 경로

앞에서 손으로 만든 기록에는 `/requirements-interview/SKILL.md`라는 경로를 썼습니다. 실제 실행이 `skill_reads`에 남긴 경로는 어떤 모양일까요? 점수 함수는 끝부분만 보기 때문에 앞부분은 무엇이어도 되지만, 실제로 어떤 값이 들어가는지 한 번은 눈으로 봐 둘 만합니다.

In [ ]:
# [따라하기] 실제 실행이 남긴 skill_reads와 trigger_ok
for case_id, record in live_records.items():
    trigger = next(score for score in live_scores[case_id] if score.name == "trigger_ok")
    print(f"{case_id}: skill_reads = {record.skill_reads} / trigger_ok = {trigger.value} ({trigger.comment})")

### 디버깅 걷기 3: 실제 실행에서 실패를 찾아 근거까지

4절의 절차를 실제 기록에 적용합니다. 모든 실행이 통과했다면 "실패 없음"이 출력됩니다(그런 경우에도 `None`인 검사 목록을 보면 무엇을 확인하지 못했는지 알 수 있습니다). 실패가 있으면 그 점수의 단계, 이유, 관련 호출, 근거 파일 순서로 나옵니다.

In [ ]:
# [따라하기] 디버깅 걷기 3: 실제 실행의 실패를 점수에서 근거까지 거슬러 올라간다
for record in live_records.values():
    trace_failure(record)
    print()

**관찰할 것.** 이 셀의 출력은 실행할 때마다 다를 수 있습니다. 어느 실행에서 어느 점수가 `False`인지는 그날의 모델 행동에 달려 있기 때문입니다. 읽는 법은 같습니다. 단계 → 이유 → 관련 호출 → 근거 파일. 호출 기록과 근거 파일이 어긋나는 곳(호출은 성공이라는데 파일에 없다, 호출은 거절인데 파일에 있다)이 있다면 그곳이 원인입니다. 원인을 찾았다고 바로 고치지 않습니다. 고쳤을 때 **정말 좋아졌는지는 같은 사례를 반복해서 재야 알 수 있고**, 그것은 이 노트북이 아니라 개선을 다루는 07의 일입니다.

## 직접 해보기

네 가지를 해 봅니다. 해보기 2와 4는 유료 호출이라서 기본값이 꺼져 있습니다(`RUN_EXERCISE = False`, `RUN_OPTIONAL = False`). 켜기 전에 셀의 `announce` 줄이 알려 주는 규모를 확인하세요.

### 해보기 1. 내 사례 만들기

`data/cases.jsonl`의 사례와 같은 모양의 사전을 직접 쓰고, **11개 검사 중 어느 것이 켜질지** 먼저 예측한 뒤 코드의 답과 비교합니다. 이 사례는 노트북 안에서만 쓰고 `data/cases.jsonl`에는 **절대 추가하지 않습니다.** 그 파일은 평가 자료이므로 에이전트가 읽지 않게 막아 둔 파일이고(`contamination_free`가 바로 그것을 감시합니다), 내용이 바뀌면 이전 결과와 비교할 수 없게 됩니다.

예시 사례는 "권한 거부와 일시 오류가 동시에 있고, 이벤트 3개를 기대하는" 사례입니다. 직접 고쳐서 다른 조합을 만들어 보세요. 필드 이름은 앞의 표(적용 조건)에 있는 것만 의미가 있습니다. 다른 이름을 쓰면 점수 함수가 읽지 않으므로 아무 검사도 켜지지 않습니다.

In [ ]:
# [내 것 만들기] 새 dev 스타일 사례를 쓰고, 어떤 검사가 켜지는지 예측과 관찰을 비교한다
my_case = {
    "id": "my-01",                      # 이 노트북 안에서만 쓰는 이름
    "split": "dev",
    "stage": "R2",
    "kind": "transient_error",
    "turns": ["사내 비품 신청 접수 도구를 만들고 싶어요. 제약은 없음, 완료 기준은 '신청 10건이 중복 없이 접수 목록에 나온다'예요."],
    "elicitation": {},
    "fault": {"transient": {"record_answer": 1}, "permission_denied": ["save_evidence"]},
    "requires": None,
    "expect": {
        "must_call": ["start_interview", "record_answer"],
        "frozen": True,
        "event_count": 3,
        # 여기에 expect 필드를 더하거나 빼 보세요: skill_read, goal_verbatim, vague_answer,
        # evidence_linked, max_same_call, must_not_call ...
    },
}


def checks_that_apply(case):
    """사례만 보고(표의 규칙으로) 켜지는 검사 이름들. 실제 점수 함수의 반응과 비교하려고 만든다."""
    return [name for name, (_, rule) in APPLIES_WHEN.items() if rule(case)]


predicted_names = checks_that_apply(my_case)
probe = make_record(my_case, empty_evidence(), calls=[one_mcp_call])
observed_names = [score.name for score in score_run(probe) if score.name in APPLIES_WHEN and score.value is not None]

print("표의 규칙으로 켜지는 검사:", predicted_names)
print("점수 함수에서 None이 아닌 검사:", observed_names)
print("서로 다른 검사:", sorted(set(predicted_names) ^ set(observed_names)))
print("이 사례가 켜지 못하는 검사:", [name for name in CHECK_NAMES if name not in observed_names])

**관찰할 것.** "서로 다른 검사"가 비어 있어야 합니다. 비어 있지 않다면 그 필드 이름의 철자나 값의 모양(예: `frozen`은 불리언)이 점수 함수가 읽는 것과 다른 것입니다. 마지막 줄 "켜지지 못하는 검사"는 이 사례로는 **볼 수 없는 것**의 목록입니다. 좋은 사례 세트는 개별 사례가 모든 것을 보는 것이 아니라, 세트 전체가 11개 검사를 각각 최소 한 번은 켜는 것입니다. 사례 세트가 목표 지표와 가드 지표를 모두 재도록 짜였는지는 `tests/test_case_coverage.py`가 분할별로 확인합니다.

### 해보기 2. 내 사례를 기준선으로 실행하기 (유료, 기본 꺼짐)

방금 만든 사례를 실제 에이전트로 한 번 돌립니다. 켜려면 아래 셀의 `RUN_EXERCISE`를 `True`로 바꾸세요. 규모는 Claude 약 9회, 실행 1회입니다. 사례에 `fault`가 있어서 서버가 장애를 주입합니다. 결과는 1회 관찰이므로 점수가 `False`여도 "에이전트가 못한다"가 아니라 "이 실행에서 이 점수가 이 이유로 나왔다"입니다.

In [ ]:
# [바꿔보기] 내 사례를 기준선 에이전트로 한 번 실행한다 (유료: RUN_EXERCISE를 True로 바꿀 때만)
RUN_EXERCISE = False

if RUN_EXERCISE:
    announce("해보기 2: 내 사례 1회 실행", model_calls=9, runs=1)
    my_record = await run_case(my_case, variant=Variant(name="baseline"), settings=load_settings(),
                               workdir=new_workdir("lab06-mine"), repeat=1)
    trace_failure(my_record)
else:
    print("RUN_EXERCISE = False 이므로 호출하지 않았습니다.")

### 해보기 3. 중복 save_evidence는 어떤 점수를 움직이는가

서버는 같은 산출물(제목, 본문, 원본 근거 ID가 같음)을 다시 받으면 같은 ID의 파일로 돌려줍니다(`cap_lab/server/evidence.py:203`). 그렇다면 에이전트가 `save_evidence`를 같은 인자로 다섯 번 불렀을 때 근거 폴더는 어떻게 되고, 11개 검사 중 무엇이 그 사실을 알아챌까요? 아래 셀에서 먼저 예측한 뒤 실행하세요.

In [ ]:
# [바꿔보기] 같은 산출물을 5번 저장해도 근거는 한 파일, 어느 검사가 알아채는가
workflow_s, saved_dir = new_lab("duplicate-save")
saved_session = interview(workflow_s, "출장 영수증 정산 도구", constraints="없음", criteria=CLEAR, freeze=True)
store_s = EvidenceStore(saved_dir)
for _ in range(5):
    store_s.save(saved_session, "최종 요약", "정산 도구 요구사항 최종 요약")  # 서버 쪽 저장 코드를 5번 호출

duplicate_case = make_case("hand-dup", {"event_count": 4, "evidence_linked": False, "must_call": ["save_evidence"]})
five_saves = [tool_call("save_evidence", {"session_id": saved_session, "title": "최종 요약",
                                         "content": "정산 도구 요구사항 최종 요약"}) for _ in range(5)]
duplicate_record = make_record(duplicate_case, saved_dir, calls=five_saves)

artifact_files = sorted(path.name for path in saved_dir.glob("*/artifact-*.json"))
print("artifact 파일 수:", len(artifact_files), "/ save_evidence 호출 수:", len(five_saves))
show_table([{"검사": score.name, "값": score.value, "이유(comment)": score.comment}
            for score in score_run(duplicate_record) if score.name in ("side_effect_stable", "tool_selection_ok", "evidence_linked")])
print("tool_calls 숫자:", next(score.value for score in score_run(duplicate_record) if score.name == "tool_calls"))

**관찰할 것.** 파일은 하나인데 호출은 다섯 번입니다. `side_effect_stable`은 통과입니다. 이 검사는 **이벤트**만 세고 산출물(artifact) 파일은 세지 않기 때문입니다(`EVENT_KINDS`, `cap_lab/scorers.py:24`). 중복 저장은 이 노트북의 11개 검사로는 `tool_calls` 숫자 외에는 잡히지 않는 사각지대입니다. 서버가 중복을 흡수해서 부작용이 없다는 사실과, 에이전트가 불필요하게 같은 호출을 되풀이했다는 사실은 별개입니다. 후자를 평가하고 싶다면 어떤 점수가 필요할까요? 새 점수를 사전 등록 없이 만들어 끼워 넣는 것은 하지 않고, 필요한 점수의 이름과 계산 방법만 글로 적어 보세요.

### 해보기 4. 반복해서 분포를 보기 (유료, 기본 꺼짐)

5절의 실행은 1회 관찰이었습니다. dev-10을 같은 기준선으로 **3회 반복**하면 `retry_bounded`가 매번 같은 값인지 볼 수 있습니다. 규모는 Claude 약 27회, 실행 3회입니다. 켜려면 `RUN_OPTIONAL`을 `True`로 바꾸세요. 결과가 3회 모두 같더라도 그것은 "이 3회에서 같았다"입니다. `pass_rate_table`이 Wilson 구간을 같이 보여 주므로, 구간이 얼마나 넓은지 확인해 보세요.

In [ ]:
# [바꿔보기] dev-10을 3회 반복해 retry_bounded가 매번 같은지 본다 (유료: RUN_OPTIONAL을 True로 바꿀 때만)
RUN_OPTIONAL = False

if RUN_OPTIONAL:
    announce("해보기 4: dev-10 3회 반복", model_calls=27, runs=3)
    repeat_workdir = new_workdir("lab06-repeat")
    repeat_rows = []
    for repeat in range(1, 4):
        record = await run_case(cases_by_id["dev-10"], variant=Variant(name="baseline"), settings=load_settings(),
                                workdir=repeat_workdir, repeat=repeat)
        by_name = {score.name: score.value for score in score_run(record)}
        repeat_rows.append({"variant": "baseline", "repeat": repeat, "retry_bounded": by_name["retry_bounded"],
                            "frozen_ok": by_name["frozen_ok"]})
    show_table(repeat_rows)
    show_table(pass_rate_table(repeat_rows, ["retry_bounded", "frozen_ok"]))
else:
    print("RUN_OPTIONAL = False 이므로 호출하지 않았습니다.")

## 확인 질문

답을 말로 설명할 수 있는지 확인합니다. 정답을 적어 두지 않았습니다. 막히면 해당 절의 셀을 다시 실행해서 출력을 근거로 설명해 보세요.

1. `tool_selection_ok`가 `True`인데 `args_valid`가 `False`인 실행은 어떤 상황입니까? 그 반대(`tool_selection_ok`는 `False`, `args_valid`는 `True`)는 어떤 상황입니까?
2. 한 검사 값이 `None`인 것과 `False`인 것은 무엇이 다릅니까? `None`을 실패로 세면 통과율이 어떻게 바뀌고, `None`을 빼고 세면서 아무 말도 하지 않으면 무엇이 문제입니까?
3. Skill을 정확히 읽었는데도 결과가 나쁜 경우와, Skill을 안 읽었는데도 결과가 좋은 경우를 하나씩 설명하세요. 이 둘을 하나의 점수로 합치면 무엇을 잃습니까?
4. `first_failure`가 `None`인데 실행이 실패한 경우를 두 가지 말해 보세요. (힌트: R 단계, 그리고 실행 오류)
5. `no_blind_retry`는 같은 인자의 호출 횟수를 셉니다. 이 방식이 놓치는 행동을 하나 들고, 그것을 어떤 숫자 점수가 보여 주는지 말하세요.
6. 서버가 중복 요청을 흡수해서 부작용이 없는 도구(`save_evidence`)와, 호출마다 새 상태를 만드는 도구(`start_interview`)가 `side_effect_stable`에 어떻게 다르게 비칩니까?
7. `contamination_free`가 `False`일 때 바로 "에이전트가 평가 자료를 훔쳐봤다"고 결론 내리면 안 되는 이유는 무엇입니까? 무엇을 먼저 확인합니까?
8. 1회 관찰과 3회 이상 반복한 결과는 무엇이 다릅니까? 5절의 점수표에서 말할 수 있는 것과 말할 수 없는 것을 하나씩 말하세요.

## 정리

- 능력 수준의 결함은 **단계로 가릅니다.** S1(Trigger), S2(도구 선택, 인자, 목표 유지), S3(모호함 처리), S5(확정), S6(근거 연결)와 견고성 R1(권한 거부), R2(일시 오류), R3(중복 실행)입니다. 점수 함수는 호출 기록과 근거 파일만 읽는 결정적 코드이고 모델의 판정을 쓰지 않습니다.
- `None`은 "확인하지 않음 또는 해당 없음"이지 실패가 아닙니다. 표에는 통과 수, 확인한 수, 확인하지 못한 수를 함께 적습니다.
- Trigger Eval과 Quality Eval은 **따로 계산하고 따로 보고합니다.** 합친 숫자와 `first_failure`만 보면 한 쪽 실패가 다른 쪽 뒤에 숨습니다.
- 점수 함수도 코드이고 가정이 있습니다. 검증 오류의 모양(`ok=False`, 목록), 경로의 끝부분, 같은 인자, `/data/` 문자열이 그 예입니다. 가정이 깨지는 곳(깨뜨리기 셀 4개)을 알아야 점수를 정확히 읽을 수 있습니다.
- 디버깅은 같은 다섯 걸음입니다. 점수표의 `False` → `first_failure`(R 단계는 따로) → comment → 관련 호출 → 근거 파일. 호출 기록은 에이전트가 한 일이고, 근거 파일은 서버가 받은 일입니다.
- 이 노트북의 실제 실행은 1회 관찰입니다. 원인을 찾은 것과 고친 것이 좋아졌는지 재는 것은 다른 일이고, 후자는 반복과 비교 설계가 필요합니다.